# RFM через groupby


Работаем как команда CRM маркетплейса: из трёх связанных таблиц нужно получить
объяснимые признаки, а не просто добиться вывода без ошибки. Перед каждой
операцией сформулируйте единицу наблюдения, ключ соединения и ожидаемое число
строк. После операции прочитайте assert как исполняемый контракт.

Сначала сделайте минимальный рабочий вариант, затем проверьте его на данных и
только после этого интерпретируйте результат. Не вводите метку churn: в этом
модуле мы строим и проверяем признаки, но не обучаем модель оттока.


**Центральная идея:** RFM переводит строки заказов в одну строку клиента по заранее зафиксированным агрегатам.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

def find_csv(name):
    for path in (Path(name), Path("../../data") / name):
        if path.exists():
            return path.resolve()
    raise FileNotFoundError(f"{name} не найден рядом с ноутбуком или в ../../data")

orders = pd.read_csv(find_csv("orders_slim.csv"), parse_dates=["order_purchase_timestamp", "order_delivered_customer_date"])
customers = pd.read_csv(find_csv("customers_slim.csv"))
payments = pd.read_csv(find_csv("payments_slim.csv"))
assert len(orders) and len(customers) and len(payments)
assert orders["order_id"].is_unique and customers["customer_id"].is_unique
print(f"orders={len(orders)}, customers={len(customers)}, payments={len(payments)}")


## 1. Проверка ключей

Проверьте уникальность ключей и покрытие оплат.

**Зачем:** RFM переводит строки заказов в одну строку клиента по заранее зафиксированным агрегатам. Зафиксируйте ожидаемую форму результата до запуска. Если assert падает, сравните единицу наблюдения до и после операции; не удаляйте проверку и не подгоняйте константу под случайный вывод.

In [ ]:
key_checks={"orders":None,"customers":None,"payments":None}  # TODO
assert set(key_checks.values())=={True}


## 2. Таблица заказ+оплата

Сделайте one-to-one join.

**Зачем:** RFM переводит строки заказов в одну строку клиента по заранее зафиксированным агрегатам. Зафиксируйте ожидаемую форму результата до запуска. Если assert падает, сравните единицу наблюдения до и после операции; не удаляйте проверку и не подгоняйте константу под случайный вывод.

In [ ]:
merged=None  # TODO
assert len(merged)==3500 and merged["payment_value"].notna().all()


## 3. Опорная дата

Используйте максимум даты всей таблицы.

**Зачем:** RFM переводит строки заказов в одну строку клиента по заранее зафиксированным агрегатам. Зафиксируйте ожидаемую форму результата до запуска. Если assert падает, сравните единицу наблюдения до и после операции; не удаляйте проверку и не подгоняйте константу под случайный вывод.

In [ ]:
ref_date=None  # TODO
assert ref_date==orders["order_purchase_timestamp"].max()
REF_NOTE=""  # TODO, >=160
assert len(REF_NOTE)>=160


## 4. Frequency

Посчитайте число уникальных order_id на клиента.

**Зачем:** RFM переводит строки заказов в одну строку клиента по заранее зафиксированным агрегатам. Зафиксируйте ожидаемую форму результата до запуска. Если assert падает, сравните единицу наблюдения до и после операции; не удаляйте проверку и не подгоняйте константу под случайный вывод.

In [ ]:
frequency=None  # TODO: Series
assert len(frequency)==778 and int(frequency.sum())==3500
assert frequency.ge(1).all()


## 5. Monetary

Посчитайте сумму payment_value.

**Зачем:** RFM переводит строки заказов в одну строку клиента по заранее зафиксированным агрегатам. Зафиксируйте ожидаемую форму результата до запуска. Если assert падает, сравните единицу наблюдения до и после операции; не удаляйте проверку и не подгоняйте константу под случайный вывод.

In [ ]:
monetary=None  # TODO
assert len(monetary)==778 and np.isclose(monetary.sum(),payments["payment_value"].sum())
assert monetary.gt(0).all()


## 6. Recency

Посчитайте дни от последней покупки до ref_date.

**Зачем:** RFM переводит строки заказов в одну строку клиента по заранее зафиксированным агрегатам. Зафиксируйте ожидаемую форму результата до запуска. Если assert падает, сравните единицу наблюдения до и после операции; не удаляйте проверку и не подгоняйте константу под случайный вывод.

In [ ]:
last_purchase=None; recency=None  # TODO
assert len(recency)==778 and recency.ge(0).all()
assert recency.min()==0


## 7. Сборка RFM

Объедините три Series в DataFrame.

**Зачем:** RFM переводит строки заказов в одну строку клиента по заранее зафиксированным агрегатам. Зафиксируйте ожидаемую форму результата до запуска. Если assert падает, сравните единицу наблюдения до и после операции; не удаляйте проверку и не подгоняйте константу под случайный вывод.

In [ ]:
rfm=None  # TODO
assert list(rfm.columns)==["customer_id","Recency","Frequency","Monetary"]
assert len(rfm)==778 and rfm["customer_id"].is_unique


## 8. Профиль и интерпретация

Получите describe и объясните высокий M при низком F.

**Зачем:** RFM переводит строки заказов в одну строку клиента по заранее зафиксированным агрегатам. Зафиксируйте ожидаемую форму результата до запуска. Если assert падает, сравните единицу наблюдения до и после операции; не удаляйте проверку и не подгоняйте константу под случайный вывод.

In [ ]:
stats=None; top5=None  # TODO
assert stats.shape==(8,3) and len(top5)==5 and top5["Monetary"].is_monotonic_decreasing
RFM_NOTE=""  # TODO >=220
assert len(RFM_NOTE)>=220
